In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time

# Check Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define Hyper-parameters 
input_size = 784
hidden_size = 500
num_classes = 10
num_epochs = 20
batch_size = 100
learning_rate = 0.01

In [ ]:
# MNIST dataset 
train_dataset = torchvision.datasets.MNIST(root='./data', 
                                           train=True, 
                                           transform=transforms.ToTensor(),  
                                           download=True)

test_dataset = torchvision.datasets.MNIST(root='./data', 
                                          train=False, 
                                          transform=transforms.ToTensor())

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, 
                                          batch_size=batch_size, 
                                          shuffle=False)

In [ ]:
# 4 layer network
class NeuralNet1(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet1, self).__init__()
        self.fc1 = nn.Linear(input_size, 20) 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 50)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(50, 20)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(20, 10) # Output neuron
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)

        return out

# 6 layer network
class NeuralNet2(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet2, self).__init__()
        self.fc1 = nn.Linear(input_size, 10) 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 20)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(20, 30)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(30, 20)
        self.relu = nn.ReLU()
        self.fc5 = nn.Linear(20, 10)
        self.relu = nn.ReLU()
        self.fc6 = nn.Linear(10, 10) # Output neuron
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.relu(out)
        out = self.fc5(out)
        out = self.relu(out)
        out = self.fc6(out)
        return out

# 6 layer network
class NeuralNet3(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet3, self).__init__()
        self.fc1 = nn.Linear(input_size, 10) 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 40)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(40, 70)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(70, 40)
        self.relu = nn.ReLU()
        self.fc5 = nn.Linear(40, 10)
        self.relu = nn.ReLU()
        self.fc6 = nn.Linear(10, 10) # Output neuron
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.relu(out)
        out = self.fc5(out)
        out = self.relu(out)
        out = self.fc6(out)
        return out

In [ ]:
networks = [NeuralNet1, NeuralNet2, NeuralNet3]

total_time = []
accuracy = []

# Takes each neural network and trains it for 20 epochs with a learning rate of 0.01
for network in networks:
    start = time.time() # Used to measure how long it takes for each model to be trained

    model = network(input_size, hidden_size, num_classes).to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  

    # Train the model
    total_step = len(train_loader)
    for epoch in range(num_epochs):
        for i, (images, labels) in enumerate(train_loader):  
            # Move tensors to the configured device
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
        
            # Backprpagation and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
            if (i+1) % 100 == 0:
                print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                       .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

    # Test the model
    # In the test phase, don't need to compute gradients (for memory efficiency)
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
        print('Accuracy of the network on the test images: {} %'.format(100 * correct / total))

        accuracy.append(100 * correct / total)

        end = time.time()
        total_time.append(end - start) # Measures how long it took for the current model to be trained

net1_accuracy = accuracy[0]
net2_accuracy = accuracy[1]
net3_accuracy = accuracy[2]

In [ ]:
print(f"Time to train model 1: {total_time[0]}s")
print(f"Time to train model 2: {total_time[1]}s")
print(f"Time to train model 3: {total_time[2]}s")

# Determines which model had the least amount of error for validation based on which model had the highest accuracy.
if net1_accuracy > net2_accuracy and net1_accuracy > net3_accuracy:
    print(f"The model with the least amount of error for validation is model 1 with an accuracy of {net1_accuracy}%")
elif net2_accuracy > net1_accuracy and net2_accuracy > net3_accuracy:
    print(f"The model with the least amount of error for validation is model 2 with an accuracy of {net2_accuracy}%")
else:
    print(f"The model with the least amount of error for validation is model 3 with an accuracy of {net3_accuracy}%")